# Code Evaluation: Benchmarking Models

## Learning Objectives

In [ ]:
import numpy as np
from collections import defaultdict

## Level 1: Execution-Based Evaluation

In [ ]:
def test_code(code: str, tests):
    try:
        compile(code, '<string>', 'exec')
    except:
        return {'passed': False, 'mode': 'syntax'}
    num_passed = 0
    for inp, exp in tests:
        try:
            local = {}
            exec(code, {}, local)
            func = next((v for k, v in local.items() if callable(v)), None)
            if func:
                args = eval(f'[{inp}]') if inp else []
                if str(func(*args)) == str(exp):
                    num_passed += 1
        except:
            pass
    return {'passed': num_passed == len(tests), 'num': num_passed, 'total': len(tests)}

code = 'def add(a, b):\n    return a + b'
tests = [('1, 2', '3'), ('5, 5', '10')]
r = test_code(code, tests)
print(f'Result: {r}')

In [ ]:
print('Test 1 passed')

## Level 2: Pass@k Metric

In [ ]:
class PassAtK:
    def compute(self, problems, k_vals):
        results = {}
        for k in k_vals:
            pass_at_k = [1.0 if any(p[:k]) else 0.0 for p in problems]
            mean = np.mean(pass_at_k)
            results[f'pass@{k}'] = mean
        return results

calc = PassAtK()
probs = [[True, False, True], [False, False, False], [True, True, True]]
r = calc.compute(probs, [1, 2, 3])
for k, v in r.items():
    print(f'{k}: {v:.1%}')

In [ ]:
print('Pass@k computed')

## Real-World Example 1: HumanEval

In [ ]:
class HumanEvalSim:
    def __init__(self, n=50):
        self.problems = [{'p': np.random.uniform(0.2, 0.8)} for _ in range(n)]
    def evaluate(self, k=1):
        results = []
        for p in self.problems:
            samples = [np.random.random() < p['p'] for _ in range(k)]
            results.append(samples)
        pass_1 = np.mean([r[0] for r in results])
        pass_k = np.mean([1.0 if any(r) else 0.0 for r in results])
        return {'pass@1': pass_1, f'pass@{k}': pass_k}

sim = HumanEvalSim(50)
r = sim.evaluate(10)
print(f'Pass@1: {r["pass@1"]:.1%}, Pass@10: {r["pass@10"]:.1%}')

## Real-World Example 2: Failure Analysis

In [ ]:
class FailureAnalyzer:
    def analyze(self, results):
        dist = defaultdict(int)
        for r in results:
            mode = 'correct' if r.get('passed') else r.get('mode', 'logic')
            dist[mode] += 1
        total = len(results)
        return {k: v/total for k, v in dist.items()}

mock = [{'passed': True}, {'passed': False, 'mode': 'syntax'}, {'passed': False, 'mode': 'logic'}]
a = FailureAnalyzer().analyze(mock)
for mode, pct in a.items():
    print(f'{mode}: {pct:.0%}')

## Key Takeaways

In [ ]:
print('Code evaluation complete')